In [31]:
from telethon import TelegramClient
import pandas as pd
from telethon.errors import FloodWaitError
import asyncio
import os
import json

api_id = "27406534"
api_hash = '06dfedef1293c5a357b1a5a9cf29de49'
phone_number = '+393386019368'
client = TelegramClient('session_name', api_id, api_hash)

async def connect_client():
    await client.start(phone=phone_number)
    print("Client connected")
    
async def shutdown_client():
    try:
        await client.disconnect()
    except:
        pass

In [26]:
async def get_entity(entity_name):
    try:
        entity = await client.get_entity(entity_name)
        print(f"Accessing {entity_name}")
        return entity
    except Exception as e:
        print(f"Failed to access {entity_name}: {e}")
        
async def get_message_count(entity_name):
    entity = await get_entity(entity_name)
    if entity:
        try:
            total_messages = await client.get_messages(entity, limit=0)
            print(f"Total messages in {entity_name}: {total_messages.total}")
            return total_messages.total
        except Exception as e:
            print(f"Failed to get message count for {entity_name}: {e}")
            return 0
    return 0

In [33]:
async def scrape_messages(entity_name, limit=100, save_media=True):
    messages = []
    entity = await get_entity(entity_name)
    print(f"Scraping messages from {entity_name}")
    if save_media:
        os.makedirs("data/media", exist_ok=True)
    async for message in client.iter_messages(entity, limit=limit):
        try:
            media_path = None
            media_url = None
            views = message.views if message.views is not None else 0
            reaction_details = {}
            if message.reactions and message.reactions.results:
                for reaction in message.reactions.results:
                    emoji = getattr(reaction.reaction, 'emoticon', str(reaction.reaction))
                    reaction_details[emoji] = reaction_details.get(emoji, 0) + reaction.count
            reaction_count = sum(reaction_details.values())
            reaction_rendered = " | ".join(f"{emoji} x{count}" for emoji, count in reaction_details.items()) if reaction_details else None
            if message.media:
                media_url = f"https://t.me/{entity_name}/{message.id}"
                if save_media:
                    media_path = await client.download_media(message, file=f"data/media/{message.id}")
            messages.append({
                'message_id': message.id,
                'date': message.date,
                'sender_id': message.sender_id,
                'message': message.message,
                'media_path': media_path,
                'media_url': media_url,
                'views': views,
                'reaction_count': reaction_count,
                'reaction_breakdown': json.dumps(reaction_details, ensure_ascii=False) if reaction_details else None,
                'reaction_rendered': reaction_rendered
            })
        except FloodWaitError as e:
            print(f"Rate limit hit, sleeping for {e.seconds} seconds")
            await asyncio.sleep(e.seconds)
    df = pd.DataFrame(messages)
    df.to_csv(f'data/{entity_name}_messages.csv', index=False)


In [28]:
user = "jungenationalisten"
saveMedia = False 
maxMessages = 50


In [32]:
import nest_asyncio
nest_asyncio.apply()

async def main():
    await connect_client()
    
    total_count = await get_message_count(user)
    
    scrape_limit = min(total_count, maxMessages) if total_count else maxMessages 
    print(scrape_limit)

    await scrape_messages(user, limit=scrape_limit, save_media=saveMedia)
    await shutdown_client()
    
loop = asyncio.get_event_loop()
loop.run_until_complete(main())


Client connected
Accessing jungenationalisten
Total messages in jungenationalisten: 4030
50
Accessing jungenationalisten
Scraping messages from jungenationalisten
